In [2]:
import numpy as np
import pyroomacoustics as pra
import matplotlib.pyplot as plt

room_dim = [3.1, 4.1, 2.4]       #размер комнаты
fs = 8000

room = pra.ShoeBox(room_dim, fs=fs)          #создаю "комнату"

source_loc = [2.0, 3.0, 1.5]
signal = np.random.rand(4*fs)  #белый шум
room.add_source(source_loc, signal=signal)

mic_positions = np.array([         #формирую микрофонную решетку
    [1.11],   #X
    [0.45],   #Y
    [1.2]     #Z
    ])
room.add_microphone_array(pra.MicrophoneArray(mic_positions, room.fs))           #добавляю решетку в комнату

room.simulate()
signals = room.mic_array.signals

#print(signals)



STFT — универсальный инструмент для обработки звуковых сигналов, который позволяет определять сложную амплитуду в зависимости от времени и частоты для любого сигнала.
https://ru.wikipedia.org/wiki/Оконное_преобразование_Фурье
Представлен устарелый вариант, был получен при использовании AI для представления дальнейших работ (возможно использование ShortTimeFFT)

In [ ]:
"""
from scipy.signal import stft

nperseg = ?
f, t, X = stft(signals, nperseg=nperseg)
X = X.transpose(2, 0, 1)

time_idx = ?
freq_idx = ?
X_freq = X[time_idx, freq_idx, :]

X_freq = X_freq.reshape(-1, 1)
"""

In [4]:
def ESPRIT(signal_matrix, num_source=1):
    
    M,T = signal_matrix.shape

    X = signal_matrix @ signal_matrix.conj().T/T  #ковариац матрица
    _, Y = np.linalg.eigh(X) #раскладываю по собю значениям
    Ys = Y[:,:-num_source] #выделяю подпространство

    
    Ys1 = Ys[:-1, :]
    Ys2 = Ys[1:, :]

    A = np.linalg.pinv(Ys1) @ Ys2

    eigvals = np.linalg.eigvals(A)  #собсственная матрица
    angle = np.angle(eigvals)  #собственные значения

    return angle * 180 / np.pi

angles = ESPRIT(X_freq)
print(angles)
estimated_angle = angles[0]

plt.figure()

plt.plot([estimated_angle,estimated_angle], [0,1])
plt.title("")
plt.xlabel("Угол")

plt.tight_layout()
plt.show()


NameError: name 'X_freq' is not defined